## Imports 

In [1]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from shapely.geometry import Point

## Maps and Dicts to Use Later

In [2]:
with open('../data/mapping_files/name_to_fips.pkl', 'rb') as f:
    name_to_fips = pickle.load(f)
    print(name_to_fips)

zip_map = pd.read_csv("../data/mapping_files/zip_to_county_mapping.csv", dtype={"zip_code": str, "county_fips": str})
zip_to_county_dict = dict(zip(zip_map["zip_code"], zip_map["county_fips"]))


data = {
    'FIPS': [
        '06001', '06003', '06005', '06007', '06009', '06011', '06013', '06015', '06017', '06019',
        '06021', '06023', '06025', '06027', '06029', '06031', '06033', '06035', '06037', '06039',
        '06041', '06043', '06045', '06047', '06049', '06051', '06053', '06055', '06057', '06059',
        '06061', '06063', '06065', '06067', '06069', '06071', '06073', '06075', '06077', '06079',
        '06081', '06083', '06085', '06087', '06089', '06091', '06093', '06095', '06097', '06099',
        '06101', '06103', '06105', '06107', '06109', '06111', '06113', '06115'
    ],
}

state_to_abbr = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "Puerto Rico": "PR",
    "Guam": "GU",
    "American Samoa": "AS",
    "U.S. Virgin Islands": "VI",
    "Northern Mariana Islands": "MP"
}

# Create the DataFrame
df_ca_counties = pd.DataFrame(data)

DER_THRESHOLD = 5

{'alabama': '1000', 'autauga': '1001', 'baldwin': '13009', 'barbour': '54001', 'bibb': '13021', 'blount': '47009', 'bullock': '1011', 'butler': '42019', 'calhoun': '54013', 'chambers': '48071', 'cherokee': '48073', 'chilton': '1021', 'choctaw': '40023', 'clarke': '51043', 'clay': '54015', 'cleburne': '5023', 'coffee': '47031', 'colbert': '1033', 'conecuh': '1035', 'coosa': '1037', 'covington': '28031', 'crenshaw': '1041', 'cullman': '1043', 'dale': '1045', 'dallas': '48113', 'dekalb': '47041', 'elmore': '16039', 'escambia': '12033', 'etowah': '1055', 'fayette': '54019', 'franklin': '53021', 'geneva': '1061', 'greene': '51079', 'hale': '48189', 'henry': '51089', 'houston': '48225', 'jackson': '55053', 'jefferson': '55055', 'lamar': '48277', 'lauderdale': '47097', 'lawrence': '47099', 'lee': '51105', 'limestone': '48293', 'lowndes': '28087', 'macon': '47111', 'madison': '51113', 'marengo': '1091', 'marion': '54049', 'marshall': '54051', 'mobile': '1097', 'monroe': '55081', 'montgomery': 

## Storage Data

- DER Energy Storage from: https://www.energy.ca.gov/data-reports/energy-almanac/california-electricity-data/california-energy-storage-system-survey
- Has energy storage data at the commercial, residential, and utility scale, check for census tract storage data
- Smallest granularity is zip code

In [3]:
storage = pd.read_excel("../data/raw/storage/Storage_LatLong.xlsx")

# Make names more descriptive
rename_map = {
    "Utility": "utility_name",
    "Nameplate Capacity (MW)": "storage_capacity_mw",
    "Nameplate Capacity (in KW AC)": "nameplate_capacity_kw_ac",
    "Fuel Type": "fuel_type",
    "Facility City": "city",
    "County": "county",
    "CAISO Flag": "caiso_flag",
    "Facility Zip": "zip_code",
    "Customer Sector": "sector",
    "Approval Date": "approval_date",
    'Technology Type': 'technology_type',
    'Disconnect Date': "disconnect_date",
    'OG Reported Technology Type': "og_reported_technology_type",
    'Count of Nameplate Capacity (in KW AC)': "count_of_nameplate_capacity_kw_ac",
    'Latitude (generated)': 'latitude',
    'Longitude (generated)': 'longitude',

}
storage.rename(columns=rename_map, inplace=True)
storage["storage_capacity_mw"] = storage["nameplate_capacity_kw_ac"] / 1000

# Filter only DER as being those in residential or commercial sector with less than 10 mW or 10000 kW capacity
storage_der = storage[storage["sector"].isin(["Residential", "Commercial"])]
storage_der = storage_der[storage_der["storage_capacity_mw"] <= DER_THRESHOLD]

storage_utility = storage[storage["storage_capacity_mw"] > DER_THRESHOLD]
storage_der["zip_code"] = storage_der["zip_code"].astype(str).str.zfill(5).str.strip()

### Analyzing Storage Dataset

In [4]:
print(storage_der.info())
nan_zip_rows = storage_der[storage_der['zip_code'].isna()]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")

nan_zip_pct = len(nan_zip_rows) / len(storage_der) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")
if 'storage_capacity_mw' in storage_der.columns:
    capacity_total = storage_der['storage_capacity_mw'].sum()
    capacity_lost = nan_zip_rows['storage_capacity_mw'].sum()
    print(f"\nCapacity lost if removed: {capacity_lost:.2f} kW "
          f"({capacity_lost / capacity_total * 100:.2f}% of total)")
else:
    print("\nColumn 'capacity_kw' not found, check column names for capacity.")

# # Technology type distribution overall
# print("\nTechnology mix (overall):")
# print(storage_der['fuel_type'].value_counts(normalize=True))

# # Technology mix for NaN ZIP rows
# print("\nTechnology mix (NaN ZIP only):")
# print(nan_zip_rows['fuel_type'].value_counts(normalize=True))

# Check how many unique ZIP codes are present
valid_zip_codes = storage_der.loc[~storage_der['zip_code'].isna(), 'zip_code'].nunique()
print(f"\nNumber of unique valid ZIP codes: {valid_zip_codes}")

# Check top 10 ZIP codes by capacity
if 'storage_capacity_mw' in storage_der.columns:
    print("\nTop 10 ZIP codes by total capacity:")
    print(storage_der.groupby('zip_code')['storage_capacity_mw'].sum().sort_values(ascending=False).head(10))

print(sorted(storage_der["zip_code"].unique()))

print(storage_der.head())

<class 'pandas.core.frame.DataFrame'>
Index: 2414 entries, 0 to 2563
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   count_of_nameplate_capacity_kw_ac  2414 non-null   int64  
 1   sector                             2414 non-null   object 
 2   zip_code                           2414 non-null   object 
 3   latitude                           2414 non-null   float64
 4   longitude                          2414 non-null   float64
 5   nameplate_capacity_kw_ac           2414 non-null   float64
 6   storage_capacity_mw                2414 non-null   float64
dtypes: float64(4), int64(1), object(2)
memory usage: 150.9+ KB
None

Total rows with NaN ZIP code: 0
Percentage of rows with NaN ZIP code: 0.00%

Capacity lost if removed: 0.00 kW (0.00% of total)

Number of unique valid ZIP codes: 1587

Top 10 ZIP codes by total capacity:
zip_code
93536    8.21658
92264    8.15716
92345

# Wind Data

- Wind data from: https://energy.usgs.gov/uswtdb/
- Smallest granularity is lat/long -> can be converted to census tract with shp file

In [5]:
import pandas as pd
import geopandas as gpd

uswtdb_wind = pd.read_csv("../data/raw/wind/USWTDB_wind_data.csv")

# --- rename to consistent fields (only the ones we need here) ---
uswtdb_wind = uswtdb_wind.rename(columns={
    "t_state": "state",
    "xlong": "longitude",
    "ylat": "latitude",
    "t_cap": "turbine_capacity_kw",
    "p_name": "project_name",
    "p_year": "project_year",
})

# --- keep CA turbines ---
uswtdb_wind = uswtdb_wind[uswtdb_wind["state"].eq("CA")].copy()

# --- make geodataframe of turbines ---
gdf_wind = gpd.GeoDataFrame(
    uswtdb_wind,
    geometry=gpd.points_from_xy(uswtdb_wind["longitude"], uswtdb_wind["latitude"]),
    crs="EPSG:4326"
)

# --- load ZCTAs (make sure CRS matches) ---
zip_shapes = gpd.read_file("../data/raw/boundaries/tl_2023_us_zcta520").to_crs("EPSG:4326")
zip_shapes = zip_shapes.rename(columns={"ZCTA5CE20": "zip_code"})
zip_shapes["zip_code"] = zip_shapes["zip_code"].astype(str).str.zfill(5)

# NOTE: 'within' can miss boundary points; 'intersects' is safer for point-in-polygon
joined = gpd.sjoin(
    gdf_wind,
    zip_shapes[["zip_code", "geometry"]],
    how="left",
    predicate="intersects",
)

# --- compute turbine MW and aggregate to ZIP ---
joined["turbine_mw"] = pd.to_numeric(joined["turbine_capacity_kw"], errors="coerce") / 1000.0

wind_zip = (
    joined.groupby("zip_code", as_index=False)
    .agg(
        wind_capacity_mw=("turbine_mw", "sum"),
        wind_turbine_count=("turbine_mw", "size"),
    )
)

wind_zip.to_csv("../data/processed/ca_zip_wind_capacity_from_uswtdb.csv", index=False)
print(wind_zip.head())

  zip_code  wind_capacity_mw  wind_turbine_count
0    91739              1.00                   1
1    91905             61.05                  30
2    92220              3.20                   2
3    92230              4.85                   3
4    92240             40.00                  40


### Analyzing Wind Turbine Data

In [6]:
print(wind_zip.info())
print(wind_zip.describe())
nan_zip_rows = wind_zip[wind_zip['zip_code'] == "00nan"]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")

nan_zip_pct = len(nan_zip_rows) / len(wind_zip) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")
if 'wind_capacity_mw' in wind_zip.columns:
    capacity_total = wind_zip['wind_capacity_mw'].sum()
    capacity_lost = nan_zip_rows['wind_capacity_mw'].sum()
    print(f"\nCapacity lost if removed: {capacity_lost:.2f} MW "
          f"({capacity_lost / capacity_total * 100:.2f}% of total)")
else:
    print("\nColumn 'wind_capacity_mw' not found, check column names for capacity.")

if 'wind_capacity_mw' in storage_der.columns:
    print("\nTop 10 ZIP codes by total capacity:")
    print(wind_zip.groupby('zip_code')['wind_capacity_mw'].sum().sort_values(ascending=False).head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   zip_code            42 non-null     object 
 1   wind_capacity_mw    42 non-null     float64
 2   wind_turbine_count  42 non-null     int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ KB
None
       wind_capacity_mw  wind_turbine_count
count         42.000000           42.000000
mean         106.003119          110.380952
std          256.980319          346.559251
min            0.000000            1.000000
25%            1.587500            1.250000
50%           15.445000            7.500000
75%           95.168750           61.000000
max         1542.541000         2184.000000

Total rows with NaN ZIP code: 0
Percentage of rows with NaN ZIP code: 0.00%

Capacity lost if removed: 0.00 MW (0.00% of total)


# Power Plant Data
- Power plant data from: https://atlas.eia.gov/datasets/eia::power-plants/explore?filters=eyJTdGF0ZSI6WyJDYWxpZm9ybmlhIl0sIkluc3RhbGxfTVciOlswLjEsNjgwOV19&location=36.025752%2C-116.778228%2C5.16
- Might have to put tags since
- Can be at the census tract level
- Historical data: https://www.eia.gov/electricity/data/eia860/

In [7]:
# --- Config ---
DER_THRESHOLD = 5 # 10  # run robustness with 10

BTM_SECTORS = [
    "Commercial Non-CHP",
    "Industrial Non-CHP",
    "Commercial CHP",
    "Industrial CHP",
]

# Technologies you might want to sum into a "BTM renewable capacity" outcome
RENEW_COLS = [
    "battery_capacity_mw",
    "biomass_capacity_mw",
    "geothermal_capacity_mw",
    "hydro_capacity_mw",
    "solar_capacity_mw",
    "wind_capacity_mw",
    "other_capacity_mw",
    # (exclude coal/natural gas/nuclear/crude if you want “renewables-ish” only)
]

# --- Load ---
power_plant = pd.read_csv("../data/raw/plants/Power_Plants.csv")

# --- Fix known bad rows (your manual corrections kept as-is) ---
power_plant["County"] = power_plant["County"].astype(str).str.strip().str.lower()

power_plant.loc[13284, ["County","Street_Address","City","Zip"]] = ["sonoma","30 Mark West Springs Rd","Santa Rosa",95403]
power_plant.loc[13300, ["County","Street_Address","City","Zip"]] = ["fresno","S Howard Ave","Riverdale",93656]
power_plant.loc[13305, ["County","Street_Address","City","Zip"]] = ["fresno","2704 S Maple Ave","Fresno",93725]
power_plant.loc[13307, ["County","Street_Address","City","Zip"]] = ["tulare","2045 N Plaza Dr","Visalia",93291]
power_plant.loc[13308, ["County","Street_Address","City","Zip"]] = ["tulare","13213 Rd 80","Tipton",93272]
power_plant.loc[13309, ["County","Street_Address","City","Zip"]] = ["tulare","531 Poplar Ave","Tipton",93272]
power_plant.loc[13310, ["County","Street_Address","City","Zip"]] = ["tulare","531 Poplar Ave","Tipton",93272]
power_plant.loc[13312, ["County","Street_Address","City","Zip"]] = ["tulare","3240 N Plaza Dr","Visalia",93291]
power_plant.loc[13314, ["County","Street_Address","City","Zip"]] = ["kern","Zerker Rd","McFarland",93250]
power_plant.loc[13323, ["County","Street_Address","City","Zip"]] = ["solano","4451 Blum Rd","Martinez",94553]
power_plant.loc[13324, ["County","Street_Address","City","Zip"]] = ["kern","1750 E Panama Ln","Bakersfield",93307]
power_plant.loc[13332, ["County","Street_Address","City","Zip"]] = ["napa","303 Green Island Rd","American Canyon",94503]
power_plant.loc[13338, ["County","Street_Address","City","Zip"]] = ["san luis obispo","9225 N River Rd","San Miguel",93451]
power_plant.loc[13341, ["County","Street_Address","City","Zip"]] = ["kern","27125 Pond Rd","Wasco",93280]
power_plant.loc[13427, ["County","Street_Address","City","Zip"]] = ["fresno","32581 W Harlan Ave","Cantua Creek",93608]
power_plant.loc[13445, ["County","Street_Address","City","Zip"]] = ["merced","7870 Hutchins Rd","Dos Palos",93620]
power_plant.loc[1939,  ["Street_Address","Zip"]] = ["501 Stampede Dam Road",96161]

# --- FIPS ---
power_plant["FIPS"] = power_plant["County"].map(name_to_fips).astype(str).str.zfill(5)

# --- Rename columns to your standardized schema ---
rename_map = {
    "X": "x_coord",
    "Y": "y_coord",
    "OBJECTID": "object_id",
    "Plant_Code": "plant_code",
    "Plant_Name": "plant_name",
    "Utility_ID": "utility_id",
    "Utility_Name": "utility_name",
    "sector_name": "sector",
    "Street_Address": "street_address",
    "City": "city",
    "County": "county",
    "State": "state",
    "Zip": "zip_code",
    "PrimSource": "primary_source",
    "source_desc": "source_description",
    "tech_desc": "technology_description",
    "Install_MW": "installed_capacity_mw",
    "Total_MW": "plant_capacity_mw",
    "Bat_MW": "battery_capacity_mw",
    "Bio_MW": "biomass_capacity_mw",
    "Coal_MW": "coal_capacity_mw",
    "Geo_MW": "geothermal_capacity_mw",
    "Hydro_MW": "hydro_capacity_mw",
    "HydroPS_MW": "hydro_pumped_storage_mw",
    "NG_MW": "natural_gas_capacity_mw",
    "Nuclear_MW": "nuclear_capacity_mw",
    "Crude_MW": "crude_oil_capacity_mw",
    "Solar_MW": "solar_capacity_mw",
    "Wind_MW": "wind_capacity_mw",
    "Other_MW": "other_capacity_mw",
    "Source": "data_source",
    "Period": "reporting_period",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "FIPS": "fips",
}
power_plant = power_plant.rename(columns=rename_map)

# --- Keep California only ---
power_plant["state"] = power_plant["state"].map(state_to_abbr)
power_plant = power_plant[power_plant["state"] == "CA"].copy()

# --- Clean ZIP code (string, 5 digits) ---
power_plant["zip_code"] = (
    power_plant["zip_code"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str[:5]
    .str.zfill(5)
)

# --- Option 1 outcome: BTM / on-site generation (sector-based) ---
power_plant_der = power_plant[
    (power_plant["sector"].isin(BTM_SECTORS)) &
    (power_plant["plant_capacity_mw"].notna()) &
    (power_plant["plant_capacity_mw"] <= DER_THRESHOLD)
].copy()

# Build optional renewable-ish capacity per plant row
for c in RENEW_COLS:
    if c not in power_plant_der.columns:
        power_plant_der[c] = 0.0

power_plant_der["btm_renewable_capacity_mw"] = power_plant_der[RENEW_COLS].fillna(0).sum(axis=1)

CONTROL_SECTORS = ["Electric Utility", "IPP Non-CHP"]  # + "IPP CHP" optional
power_plant_controls = power_plant[power_plant["sector"].isin(CONTROL_SECTORS)].copy()
power_plant_controls_zip = power_plant_controls.groupby("zip_code", as_index=False).agg(
    grid_supply_capacity_mw=("plant_capacity_mw", "sum"),
    grid_supply_plant_count=("plant_code", "nunique"),
)

### Analyzing Power Plant Data

In [8]:
print(power_plant_controls.info())
nan_zip_rows = power_plant_controls[power_plant_controls['zip_code'] == "00nan"]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")

nan_zip_pct = len(nan_zip_rows) / len(power_plant_controls) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")
if 'plant_capacity_mw' in power_plant_controls.columns:
    capacity_total = power_plant_controls['plant_capacity_mw'].sum()
    capacity_lost = nan_zip_rows['plant_capacity_mw'].sum()
    print(f"\nCapacity lost if removed: {capacity_lost:.2f} MW "
          f"({capacity_lost / capacity_total * 100:.2f}% of total)")
else:
    print("\nColumn 'plant_capacity_mw' not found, check column names for capacity.")


if 'plant_capacity_mw' in power_plant_controls.columns:
    print("\nTop 10 ZIP codes by total capacity:")
    print(power_plant_controls.groupby('zip_code')['plant_capacity_mw'].sum().sort_values(ascending=False).head(10))

print(sorted(power_plant_controls["zip_code"].unique()))

print(power_plant_controls.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1484 entries, 19 to 13338
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   x_coord                  1484 non-null   float64
 1   y_coord                  1484 non-null   float64
 2   object_id                1484 non-null   int64  
 3   plant_code               1484 non-null   int64  
 4   plant_name               1484 non-null   object 
 5   utility_id               1484 non-null   int64  
 6   utility_name             1483 non-null   object 
 7   sector                   1484 non-null   object 
 8   street_address           1469 non-null   object 
 9   city                     1480 non-null   object 
 10  county                   1484 non-null   object 
 11  state                    1484 non-null   object 
 12  zip_code                 1484 non-null   object 
 13  primary_source           1484 non-null   object 
 14  source_description       14

# EV Cars and Chargers
- Data on EV chargers, https://www.energy.ca.gov/data-reports/energy-almanac/zero-emission-vehicle-and-infrastructure-statistics-collection/electric


In [9]:
ev_cars = pd.read_excel("../data/raw/ev_chargers/Stock_Map_County (2)_Full Data_data.xlsx")
ev_cars.rename(columns={"MAKE": "make", "MODEL": "model", "NonZEV vs ZEV": "nonzev_v_zev",
                        "CA ZIP": "zip_code", "Fuel Type ": "fuel_type"}, inplace=True)
print(ev_cars.columns)

ev_chargers = pd.read_excel('../data/raw/ev_chargers/Charger_County Map_Full Data_data_zip_lat-lon.xlsx')
ev_chargers['County'] = ev_chargers['County'].str.strip().str.lower()
ev_chargers['fips'] = ev_chargers['County'].map(name_to_fips)
ev_chargers = ev_chargers.drop(columns=["Calculation1", "For Additional info", "For Additional info (copy)"])
ev_chargers.rename(columns={"County":"county",
                            'Access': "access_type",
                            'DC Fast': "dc_fast_chargers",
                            'Level 1':"level1_chargers",
                            'Level 2':"level2_chargers",
       'Number of Chargers':"total_chargers", "ZIP":"zip_code", "City":"city", "EV Connector Types":"ev_connector_types","ID":"id", "Latitude":"latitude",
       "Longitude":"longitude", "Network":"network", "State":"state", "Station Name":"station_name", "Street Address":"street_address", "Level 1 & 2": "level1_2_chargers"}, inplace=True)
ev_chargers['fips'] = ev_chargers['fips'].astype(str).str.zfill(5)
source_zip_raw = ev_chargers["zip_code"].copy()
source_zip_clean = (
    source_zip_raw.astype("string")
    .str.strip()
    .str.extract(r"^(\d{5})(?:\.0+)?(?:-\d+)?$", expand=False)
)
invalid_source_zip = ~source_zip_clean.str.fullmatch(r"\d{5}").fillna(False)
ev_chargers_unknown_location = ev_chargers.loc[invalid_source_zip].copy()
ev_chargers_unknown_location.insert(
    0,
    "source_zip_raw",
    source_zip_raw.loc[invalid_source_zip].astype("string"),
)
ev_chargers_unknown_location.to_csv("../data/processed/ev_chargers_unknown_location.csv", index=True)
ev_chargers = ev_chargers.loc[~invalid_source_zip].copy()
ev_chargers["zip_code"] = source_zip_clean.loc[~invalid_source_zip]

print(dict(ev_chargers["zip_code"].value_counts()))
print(dict(ev_cars["zip_code"].value_counts()))

Index(['make', 'model', 'nonzev_v_zev', 'zip_code', 'fuel_type'], dtype='object')
{'94025': 374, '95054': 315, '92618': 300, '92101': 177, '94080': 167, '92802': 155, '95814': 127, 'No in': 118, '94063': 115, '94538': 112, '92612': 111, '92121': 108, '95014': 104, '90802': 102, '92617': 102, '95112': 100, '94158': 97, '94128': 97, '92626': 95, '92037': 95, '92660': 93, '95134': 91, '94086': 91, '94010': 90, '92408': 87, '90028': 86, '94304': 86, '95110': 82, '90007': 81, '91789': 80, '95035': 80, '94403': 78, '94568': 78, '90012': 76, '95389': 75, '94306': 74, '90067': 73, '90045': 73, '90232': 72, '94558': 69, '95113': 68, '90401': 66, '90405': 66, '95128': 66, '90015': 65, '92127': 62, '92603': 62, '94588': 62, '92868': 60, '94103': 60, '92130': 59, '93446': 58, '94612': 58, '92262': 58, '92614': 58, '90025': 57, '92108': 57, '90245': 56, '90404': 56, '93401': 56, '95008': 55, '92831': 55, '95630': 54, '90071': 54, '91606': 54, '91405': 53, '90017': 53, '94539': 52, '94065': 52, '957

### Filling in NaN data 

In [10]:
gdf_ev = gpd.GeoDataFrame(
    ev_chargers,
    geometry=gpd.points_from_xy(ev_chargers["longitude"], ev_chargers["latitude"]),
    crs="EPSG:4326"
)
zip_shapes = zip_shapes.to_crs(gdf_ev.crs)
joined = gpd.sjoin(gdf_ev, zip_shapes, how="left", predicate="within")
if "zip_code_right" in joined.columns:
    spatial_zip = joined["zip_code_right"]
elif "zip_code" in joined.columns:
    spatial_zip = joined["zip_code"]
else:
    spatial_zip = pd.Series(pd.NA, index=joined.index)
spatial_zip = (
    spatial_zip.astype("string")
    .str.strip()
    .str.extract(r"^(\d{1,5})(?:\.0+)?(?:-\d+)?$", expand=False)
    .str.zfill(5)
)
ev_chargers["zip_code"] = spatial_zip.fillna(ev_chargers["zip_code"])


ev_chargers.loc[892, "zip_code"] = 96120
ev_chargers.loc[893, "zip_code"] = 96120

ev_chargers.loc[1934, "zip_code"] = 92328
ev_chargers.loc[1935, "zip_code"] = 92328

ev_chargers.loc[1942, "zip_code"] = 93526

ev_chargers.loc[1965, "zip_code"] = 93203
ev_chargers.loc[1966, "zip_code"] = 93203

ev_chargers.loc[2140, "zip_code"] = 93516
ev_chargers.loc[2141, "zip_code"] = 93516
ev_chargers.loc[2142, "zip_code"] = 93516
ev_chargers.loc[2143, "zip_code"] = 93516

ev_chargers.loc[2227, "zip_code"] = 93204

ev_chargers.loc[2187, "zip_code"] = 93266
ev_chargers.loc[2188, "zip_code"] = 93266

ev_chargers.loc[7032, "zip_code"] = 95389
ev_chargers.loc[7033, "zip_code"] = 95389
ev_chargers.loc[7034, "zip_code"] = 95389
ev_chargers.loc[7035, "zip_code"] = 95389
ev_chargers.loc[7036, "zip_code"] = 95389
ev_chargers.loc[7037, "zip_code"] = 95389
ev_chargers.loc[7038, "zip_code"] = 95389
ev_chargers.loc[7039, "zip_code"] = 95389
ev_chargers.loc[7040, "zip_code"] = 95389
ev_chargers.loc[7041, "zip_code"] = 95389
ev_chargers.loc[7042, "zip_code"] = 95389
ev_chargers.loc[7043, "zip_code"] = 95389
ev_chargers.loc[7044, "zip_code"] = 95389
ev_chargers.loc[7045, "zip_code"] = 95389
ev_chargers.loc[7046, "zip_code"] = 95389
ev_chargers.loc[7047, "zip_code"] = 95389
ev_chargers.loc[7048, "zip_code"] = 95389
ev_chargers.loc[7049, "zip_code"] = 95389
ev_chargers.loc[7050, "zip_code"] = 95389
ev_chargers.loc[7051, "zip_code"] = 95389
ev_chargers.loc[7052, "zip_code"] = 95389
ev_chargers.loc[7053, "zip_code"] = 95389
ev_chargers.loc[7054, "zip_code"] = 95389
ev_chargers.loc[7055, "zip_code"] = 95389

ev_chargers.loc[9225, "zip_code"] = 92675
ev_chargers.loc[9406, "zip_code"] = 92675

ev_chargers.loc[18155, "zip_code"] = 95389
ev_chargers.loc[18163, "zip_code"] = 95389
ev_chargers.loc[18164, "zip_code"] = 95389

ev_chargers.loc[3458, "zip_code"] = 90249

ev_chargers.loc[7139, "zip_code"] = 93620
ev_chargers.loc[7140, "zip_code"] = 93620

ev_chargers.loc[7178, "zip_code"] = 96101
ev_chargers.loc[7179, "zip_code"] = 96101

ev_chargers.loc[7183, "zip_code"] = 93541
ev_chargers.loc[7184, "zip_code"] = 93541

ev_chargers.loc[11019, "zip_code"] = 92338
ev_chargers.loc[11020, "zip_code"] = 92338
# Note about this data point in Middle Farallon Island, SF, CA: 13076/7, 94122
ev_chargers.loc[13076, "zip_code"] = 94122
ev_chargers.loc[13077, "zip_code"] = 94122

ev_chargers.loc[7243, "zip_code"] = 93451

ev_chargers.loc[17432, "zip_code"] = 96125
ev_chargers.loc[17432, "zip_code"] = 96125

ev_chargers.loc[18139, "zip_code"] = 95321
ev_chargers.loc[18140, "zip_code"] = 95321

ev_chargers.loc[11206, "zip_code"] = 92364
ev_chargers.loc[11217, "zip_code"] = 92309
ev_chargers.loc[17433, "zip_code"] = 96125
ev_chargers.loc[18039, "zip_code"] = 96063

ev_chargers["zip_code"] = (
    ev_chargers["zip_code"].astype("string")
    .str.strip()
    .str.extract(r"^(\d{1,5})(?:\.0+)?(?:-\d+)?$", expand=False)
    .str.zfill(5)
)
ev_chargers = ev_chargers[ev_chargers["zip_code"].str.fullmatch(r"\d{5}").fillna(False)].copy()
ev_chargers = ev_chargers[ev_chargers["zip_code"].str.startswith("9")].copy()


### Analyzing EV Charger Data

In [11]:
print(ev_chargers.info())
nan_zip_rows = ev_chargers[ev_chargers['zip_code'] == "00nan"]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")
type_stats = ["dc_fast_chargers","level1_chargers","level2_chargers","total_chargers"]

nan_zip_pct = len(nan_zip_rows) / len(ev_chargers) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")

# charger_dict = {}
for i in type_stats:
    if i in ev_chargers.columns:
        capacity_total = ev_chargers[i].sum()
        capacity_lost = nan_zip_rows[i].sum()
        print(f"\n{i} lost if removed: {capacity_lost:.2f}"
            f"({capacity_lost / capacity_total * 100:.2f}% of total)")
    else:
        print("\nColumn 'capacity_mw' not found, check column names for capacity.")

for i in type_stats:
    print(i)
    if i in ev_chargers.columns:
        print(f"\nTop 10 ZIP codes by {i}:")
        print(ev_chargers.groupby('zip_code')[i].sum().sort_values(ascending=False).head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18573 entries, 0 to 18572
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   county              18573 non-null  object 
 1   access_type         18573 non-null  object 
 2   dc_fast_chargers    18573 non-null  int64  
 3   level1_chargers     18573 non-null  int64  
 4   level2_chargers     18573 non-null  int64  
 5   total_chargers      18573 non-null  int64  
 6   city                18573 non-null  object 
 7   ev_connector_types  18561 non-null  object 
 8   id                  18573 non-null  int64  
 9   latitude            18573 non-null  float64
 10  level1_2_chargers   18573 non-null  int64  
 11  longitude           18573 non-null  float64
 12  network             18573 non-null  object 
 13  state               18573 non-null  object 
 14  station_name        18573 non-null  object 
 15  street_address      18572 non-null  object 
 16  zip_

In [12]:
print(ev_cars.info())
nan_zip_rows = ev_cars[ev_cars['zip_code'] == "00nan"]
print(f"\nTotal rows with NaN ZIP code: {len(nan_zip_rows)}")
type_stats = ["dc_fast_chargers","level1_chargers","level2_chargers","total_chargers"]

nan_zip_pct = len(nan_zip_rows) / len(ev_cars) * 100
print(f"Percentage of rows with NaN ZIP code: {nan_zip_pct:.2f}%")

print(ev_cars.groupby('zip_code').describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 317369 entries, 0 to 317368
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   make          317369 non-null  object
 1   model         317369 non-null  object
 2   nonzev_v_zev  317369 non-null  object
 3   zip_code      317369 non-null  int64 
 4   fuel_type     317369 non-null  object
dtypes: int64(1), object(4)
memory usage: 12.1+ MB
None

Total rows with NaN ZIP code: 0
Percentage of rows with NaN ZIP code: 0.00%
          make                      model                           \
         count unique     top  freq count unique         top  freq   
zip_code                                                             
89061        1      1   Tesla     1     1      1     Model S     1   
90001      166     32  Toyota    24   166     75       Mirai     8   
90002      133     23  Toyota    20   133     58        LEAF     8   
90003      178     25   Tesla    25 

In [13]:
type_stats = ["dc_fast_chargers","level1_chargers","level2_chargers","total_chargers"]
for i in type_stats:
    print(i)
    print(ev_chargers[i].sum())
    print(ev_chargers.groupby("zip_code")[i].sum().describe())

dc_fast_chargers
16371
count    1258.000000
mean       13.013514
std        64.460016
min         0.000000
25%         0.000000
50%         4.000000
75%        16.000000
max      2213.000000
Name: dc_fast_chargers, dtype: float64
level1_chargers
692
count    1258.000000
mean        0.550079
std        12.034931
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       420.000000
Name: level1_chargers, dtype: float64
level2_chargers
162178
count      1258.000000
mean        128.917329
std        3323.864667
min           0.000000
25%           4.000000
50%          13.000000
75%          37.000000
max      117884.000000
Name: level2_chargers, dtype: float64
total_chargers
178549
count      1258.000000
mean        141.930843
std        3386.105188
min           1.000000
25%           8.000000
50%          22.000000
75%          53.000000
max      120097.000000
Name: total_chargers, dtype: float64


# Gas and Electric Company Data
- Project Data from: https://www.californiadgstats.ca.gov/downloads/
- Data archive for time series: https://www.californiadgstats.ca.gov/archives/interconnection_rule21_projects/

In [14]:
rename_map2 = {
    'Application Id': 'application_id',
    'Matched CSI Application Number': 'matched_california_solar_initiative_application_number',
    'Application Status': 'application_status',
    'Utility': 'utility',
    'Service City': 'city',
    'Service Zip': 'zip_code',
    'Service County': 'county',
    'Technology Type': 'technology_type',
    'System Size DC': 'system_size_dc',
    'System Size AC': 'system_size_ac',
    'Storage Capacity (kWh)': 'storage_capacity_kwh',
    'Storage Size (kW AC)': 'storage_size_kw_ac',
    'Inverter Size (kW AC)': 'inverter_size_kw_ac',
    'Tilt': 'tilt',
    'Azimuth': 'azimuth',
    'Mounting Method': 'mounting_method',
    'Tracking': 'tracking',
    'Customer Sector': 'sector',
    'App Received Date': 'application_received_date',
    'App Complete Date': 'application_complete_date',
    'App Approved Date': 'application_approved_date',
    'Self Installer': 'self_installer',
    'Installer Name': 'installer_name',
    'Installer Phone': 'installer_phone',
    'Installer City': 'installer_city'
}


# SCE_intercon = pd.read_csv("SCE_Interconnected_Project_Sites_2025-01-31.csv")
# SCE_intercon.rename(columns=rename_map2, inplace=True)
# SCE_intercon['county'] = SCE_intercon['county'].str.strip().str.lower()
# SCE_intercon['fips'] = SCE_intercon['county'].map(name_to_fips)
# SCE_intercon['fips'] = SCE_intercon['fips'].astype(str).str.zfill(5)
# SCE_intercon["zip_code"] = SCE_intercon["zip_code"].astype(str).str.zfill(5).str.strip()
# SCE_intercon['total_capacity_kw'] = (SCE_intercon['system_size_ac'].fillna(0) + SCE_intercon['storage_size_kw_ac'].fillna(0))
# SCE_intercon = SCE_intercon.drop(columns=['matched_california_solar_initiative_application_number', 'storage_capacity_kwh', 'storage_size_kw_ac', 'inverter_size_kw_ac', 'Third Party Owned Type', 'Third Party Name', 'Pace Financier', 'Electric Vehicle Count', 'System Output Monitoring Provider', 'Total System Cost', 'Itc Cost Basis', 'Previous Application', 'Previous Application Ids', 'VNEM, NEM-V, NEM-Agg', 'VNEM ID', 'Match Somah Application', 'Generator Model 2', 'Generator Manufacturer 2', 'Generator Quantity 2', 'Generator Model 3', 'Generator Manufacturer 3', 'Generator Quantity 3', 'Generator Model 4', 'Generator Manufacturer 4', 'Generator Quantity 4', 'Generator Model 5', 'Generator Manufacturer 5', 'Generator Quantity 5', 'Generator Model 6', 'Generator Manufacturer 6', 'Generator Quantity 6', 'Generator Model 7', 'Generator Manufacturer 7', 'Generator Quantity 7', 'Generator Model 8', 'Generator Manufacturer 8', 'Generator Quantity 8', 'Generator Model 9', 'Generator Manufacturer 9', 'Generator Quantity 9', 'Generator Model 10', 'Generator Manufacturer 10', 'Generator Quantity 10', 'Generator Model 11', 'Generator Manufacturer 11', 'Generator Quantity 11', 'Generator Model 12', 'Generator Manufacturer 12', 'Generator Quantity 12', 'Generator Model 13', 'Generator Manufacturer 13', 'Generator Quantity 13', 'Generator Model 14', 'Generator Manufacturer 14', 'Generator Quantity 14', 'Generator Model 15', 'Generator Manufacturer 15', 'Generator Quantity 15', 'Generator Model 16', 'Generator Manufacturer 16', 'Generator Quantity 16', 'Generator Model 17', 'Generator Manufacturer 17', 'Generator Quantity 17', 'Generator Model 18', 'Generator Manufacturer 18', 'Generator Quantity 18', 'Inverter Model 2', 'Inverter Manufacturer 2', 'Inverter Quantity 2', 'Inverter Model 3', 'Inverter Manufacturer 3', 'Inverter Quantity 3', 'Inverter Model 4', 'Inverter Manufacturer 4', 'Inverter Quantity 4', 'Inverter Model 5', 'Inverter Manufacturer 5', 'Inverter Quantity 5', 'Inverter Model 6', 'Inverter Manufacturer 6', 'Inverter Quantity 6', 'Inverter Model 7', 'Inverter Manufacturer 7', 'Inverter Quantity 7', 'Inverter Model 8', 'Inverter Manufacturer 8', 'Inverter Quantity 8', 'Inverter Model 9', 'Inverter Manufacturer 9', 'Inverter Quantity 9', 'Inverter Model 10', 'Inverter Manufacturer 10', 'Inverter Quantity 10', 'Inverter Model 11', 'Inverter Manufacturer 11', 'Inverter Quantity 11', 'Inverter Model 12', 'Inverter Manufacturer 12', 'Inverter Quantity 12', 'Inverter Model 13', 'Inverter Manufacturer 13', 'Inverter Quantity 13', 'Inverter Model 14', 'Inverter Manufacturer 14', 'Inverter Quantity 14', 'Inverter Model 15', 'Inverter Manufacturer 15', 'Inverter Quantity 15', 'Inverter Model 16', 'Inverter Manufacturer 16', 'Inverter Quantity 16', 'Inverter Model 17', 'Inverter Manufacturer 17', 'Inverter Quantity 17', 'Inverter Model 18', 'Inverter Manufacturer 18', 'Inverter Quantity 18', 'Inverter Model 19', 'Inverter Manufacturer 19', 'Inverter Quantity 19', 'Inverter Model 20', 'Inverter Manufacturer 20', 'Inverter Quantity 20', 'Inverter Model 21', 'Inverter Manufacturer 21', 'Inverter Quantity 21', 'Inverter Model 22', 'Inverter Manufacturer 22', 'Inverter Quantity 22', 'Inverter Model 23', 'Inverter Manufacturer 23', 'Inverter Quantity 23', 'Inverter Model 24', 'Inverter Manufacturer 24', 'Inverter Quantity 24'])

# nan_rows = SCE_intercon[SCE_intercon['zip_code'].isna()]
# print(len(nan_rows))


# SDGE_intercon = pd.read_csv("SDGE_Interconnected_Project_Sites_2025-01-31.csv")
# SDGE_intercon.rename(columns=rename_map2, inplace=True)
# SDGE_intercon['county'] = SDGE_intercon['county'].str.strip().str.lower()
# SDGE_intercon['fips'] = SDGE_intercon['county'].map(name_to_fips)
# SDGE_intercon['fips'] = SDGE_intercon['fips'].astype(str).str.zfill(5)
# SDGE_intercon["zip_code"] = SDGE_intercon["zip_code"].astype(str).str.zfill(5).str.strip()
# SDGE_intercon['total_capacity_kw'] = (SDGE_intercon['system_size_ac'].fillna(0) + SDGE_intercon['storage_size_kw_ac'].fillna(0))
# SDGE_intercon = SDGE_intercon.drop(columns=['matched_california_solar_initiative_application_number', 'storage_capacity_kwh', 'storage_size_kw_ac', 'Pace Financier', 'Total System Cost', 'Previous Application', 'Previous Application Ids', 'VNEM, NEM-V, NEM-Agg', 'VNEM ID', 'Match Somah Application', 'Generator Model 2', 'Generator Manufacturer 2', 'Generator Quantity 2', 'Generator Model 3', 'Generator Manufacturer 3', 'Generator Quantity 3', 'Generator Model 4', 'Generator Manufacturer 4', 'Generator Quantity 4', 'Generator Model 5', 'Generator Manufacturer 5', 'Generator Quantity 5', 'Generator Model 6', 'Generator Manufacturer 6', 'Generator Quantity 6', 'Generator Model 7', 'Generator Manufacturer 7', 'Generator Quantity 7', 'Generator Model 8', 'Generator Manufacturer 8', 'Generator Quantity 8', 'Generator Model 9', 'Generator Manufacturer 9', 'Generator Quantity 9', 'Generator Model 10', 'Generator Manufacturer 10', 'Generator Quantity 10', 'Generator Model 11', 'Generator Manufacturer 11', 'Generator Quantity 11', 'Generator Model 12', 'Generator Manufacturer 12', 'Generator Quantity 12', 'Generator Model 13', 'Generator Manufacturer 13', 'Generator Quantity 13', 'Generator Model 14', 'Generator Manufacturer 14', 'Generator Quantity 14', 'Generator Model 15', 'Generator Manufacturer 15', 'Generator Quantity 15', 'Generator Model 16', 'Generator Manufacturer 16', 'Generator Quantity 16', 'Generator Model 17', 'Generator Manufacturer 17', 'Generator Quantity 17', 'Generator Model 18', 'Generator Manufacturer 18', 'Generator Quantity 18', 'Inverter Model 2', 'Inverter Manufacturer 2', 'Inverter Quantity 2', 'Inverter Model 3', 'Inverter Manufacturer 3', 'Inverter Quantity 3', 'Inverter Model 4', 'Inverter Manufacturer 4', 'Inverter Quantity 4', 'Inverter Model 5', 'Inverter Manufacturer 5', 'Inverter Quantity 5', 'Inverter Model 6', 'Inverter Manufacturer 6', 'Inverter Quantity 6', 'Inverter Model 7', 'Inverter Manufacturer 7', 'Inverter Quantity 7', 'Inverter Model 8', 'Inverter Manufacturer 8', 'Inverter Quantity 8', 'Inverter Model 9', 'Inverter Manufacturer 9', 'Inverter Quantity 9', 'Inverter Model 10', 'Inverter Manufacturer 10', 'Inverter Quantity 10', 'Inverter Model 11', 'Inverter Manufacturer 11', 'Inverter Quantity 11', 'Inverter Model 12', 'Inverter Manufacturer 12', 'Inverter Quantity 12', 'Inverter Model 13', 'Inverter Manufacturer 13', 'Inverter Quantity 13', 'Inverter Model 14', 'Inverter Manufacturer 14', 'Inverter Quantity 14', 'Inverter Model 15', 'Inverter Manufacturer 15', 'Inverter Quantity 15', 'Inverter Model 16', 'Inverter Manufacturer 16', 'Inverter Quantity 16', 'Inverter Model 17', 'Inverter Manufacturer 17', 'Inverter Quantity 17', 'Inverter Model 18', 'Inverter Manufacturer 18', 'Inverter Quantity 18', 'Inverter Model 19', 'Inverter Manufacturer 19', 'Inverter Quantity 19', 'Inverter Model 20', 'Inverter Manufacturer 20', 'Inverter Quantity 20', 'Inverter Model 21', 'Inverter Manufacturer 21', 'Inverter Quantity 21', 'Inverter Model 22', 'Inverter Manufacturer 22', 'Inverter Quantity 22', 'Inverter Model 23', 'Inverter Manufacturer 23', 'Inverter Quantity 23', 'Inverter Model 24', 'Inverter Manufacturer 24', 'Inverter Quantity 24'])

# nan_rows = SDGE_intercon[SDGE_intercon['zip_code'].isna()]
# print(len(nan_rows))

In [15]:
# PGE_intercon = pd.read_csv("PGE_Interconnected_Project_Sites_2025-01-31.csv")
# PGE_intercon.rename(columns=rename_map2, inplace=True)
# PGE_intercon['county'] = PGE_intercon['county'].str.strip().str.lower()
# PGE_intercon['fips'] = PGE_intercon['county'].map(name_to_fips)
# PGE_intercon['fips'] = PGE_intercon['fips'].astype(str).str.zfill(5)
# PGE_intercon['total_capacity_kw'] = (PGE_intercon['system_size_ac'].fillna(0) + PGE_intercon['storage_size_kw_ac'].fillna(0))

# nan_rows = PGE_intercon[PGE_intercon['zip_code'].isna()]
# PGE_intercon = PGE_intercon[PGE_intercon['zip_code'].notna()]

# PGE_intercon["zip_code"] = PGE_intercon["zip_code"].astype(int).astype(str)[:5]
# PGE_intercon = PGE_intercon.drop(columns=['matched_california_solar_initiative_application_number', 'Third Party Owned Type', 'Third Party Name', 'Pace Financed', 'Pace Financier', 'System Output Reports To Vendor?', 'System Output Monitoring Provider', 'Previous Application', 'Previous Application Ids', 'VNEM, NEM-V, NEM-Agg', 'VNEM ID', 'Match Somah Application', 'Generator Model 3', 'Generator Manufacturer 3', 'Generator Quantity 3', 'Generator Model 4', 'Generator Manufacturer 4', 'Generator Quantity 4', 'Generator Model 5', 'Generator Manufacturer 5', 'Generator Quantity 5', 'Generator Model 6', 'Generator Manufacturer 6', 'Generator Quantity 6', 'Generator Model 7', 'Generator Manufacturer 7', 'Generator Quantity 7', 'Generator Model 8', 'Generator Manufacturer 8', 'Generator Quantity 8', 'Generator Model 9', 'Generator Manufacturer 9', 'Generator Quantity 9', 'Generator Model 10', 'Generator Manufacturer 10', 'Generator Quantity 10', 'Generator Model 11', 'Generator Manufacturer 11', 'Generator Quantity 11', 'Generator Model 12', 'Generator Manufacturer 12', 'Generator Quantity 12', 'Generator Model 13', 'Generator Manufacturer 13', 'Generator Quantity 13', 'Generator Model 14', 'Generator Manufacturer 14', 'Generator Quantity 14', 'Generator Model 15', 'Generator Manufacturer 15', 'Generator Quantity 15', 'Generator Model 16', 'Generator Manufacturer 16', 'Generator Quantity 16', 'Generator Model 17', 'Generator Manufacturer 17', 'Generator Quantity 17', 'Generator Model 18', 'Generator Manufacturer 18', 'Generator Quantity 18', 'Inverter Model 3', 'Inverter Manufacturer 3', 'Inverter Quantity 3', 'Inverter Model 4', 'Inverter Manufacturer 4', 'Inverter Quantity 4', 'Inverter Model 5', 'Inverter Manufacturer 5', 'Inverter Quantity 5', 'Inverter Model 6', 'Inverter Manufacturer 6', 'Inverter Quantity 6', 'Inverter Model 7', 'Inverter Manufacturer 7', 'Inverter Quantity 7', 'Inverter Model 8', 'Inverter Manufacturer 8', 'Inverter Quantity 8', 'Inverter Model 9', 'Inverter Manufacturer 9', 'Inverter Quantity 9', 'Inverter Model 10', 'Inverter Manufacturer 10', 'Inverter Quantity 10', 'Inverter Model 11', 'Inverter Manufacturer 11', 'Inverter Quantity 11', 'Inverter Model 12', 'Inverter Manufacturer 12', 'Inverter Quantity 12', 'Inverter Model 13', 'Inverter Manufacturer 13', 'Inverter Quantity 13', 'Inverter Model 14', 'Inverter Manufacturer 14', 'Inverter Quantity 14', 'Inverter Model 15', 'Inverter Manufacturer 15', 'Inverter Quantity 15', 'Inverter Model 16', 'Inverter Manufacturer 16', 'Inverter Quantity 16', 'Inverter Model 17', 'Inverter Manufacturer 17', 'Inverter Quantity 17', 'Inverter Model 18', 'Inverter Manufacturer 18', 'Inverter Quantity 18', 'Inverter Model 19', 'Inverter Manufacturer 19', 'Inverter Quantity 19', 'Inverter Model 20', 'Inverter Manufacturer 20', 'Inverter Quantity 20', 'Inverter Model 21', 'Inverter Manufacturer 21', 'Inverter Quantity 21', 'Inverter Model 22', 'Inverter Manufacturer 22', 'Inverter Quantity 22', 'Inverter Model 23', 'Inverter Manufacturer 23', 'Inverter Quantity 23', 'Inverter Model 24', 'Inverter Manufacturer 24', 'Inverter Quantity 24'])



# capacity_lost = nan_rows['total_capacity_kw'].sum()
# total_capacity = PGE_intercon['total_capacity_kw'].sum()

# percent_rows_lost = len(nan_rows) / PGE_intercon.shape[0] * 100
# percent_capacity_lost = capacity_lost / total_capacity * 100

# print(f"Rows removed: {len(nan_rows)} ({percent_rows_lost:.2f}%)")
# print(f"Capacity lost: {capacity_lost:.2f} kW ({percent_capacity_lost:.2f}%)")

# removed_stats = nan_rows['total_capacity_kw'].describe()
# kept_stats = PGE_intercon['total_capacity_kw'].describe()

# print("Removed stats:\n", removed_stats)
# print("Kept stats:\n", kept_stats)

# print("Removed tech mix:\n", nan_rows['technology_type'].value_counts(normalize=True))
# print("Kept tech mix:\n", PGE_intercon['technology_type'].value_counts(normalize=True))

# print("Removed sector mix:\n", nan_rows['sector'].value_counts(normalize=True))
# print("Kept sector mix:\n", PGE_intercon['sector'].value_counts(normalize=True))

# print("Removed cities:\n", nan_rows['city'].value_counts().head(10))

# print(PGE_intercon['zip_code'].unique())
# print(len(nan_rows))

# print(PGE_intercon['technology_type'].unique())


# print("Capacities:\n", PGE_intercon['total_capacity_kw'].value_counts().head(10))




# mask_storage = PGE_intercon['technology_type'].str.contains('Battery', case=False, na=False)
# removed_rows = PGE_intercon[mask_storage]
# kept_rows = PGE_intercon[~mask_storage]

# print(f"Rows removed: {removed_rows.shape[0]} ({removed_rows.shape[0]/len(PGE_intercon)*100:.2f}%)")

# capacity_lost = removed_rows['total_capacity_kw'].sum()
# capacity_total = PGE_intercon['total_capacity_kw'].sum()

# print(f"Capacity lost: {capacity_lost:.2f} kW ({capacity_lost/capacity_total*100:.2f}% of total)")

# print("\nRemoved technology mix:")
# print(removed_rows['technology_type'].value_counts(normalize=True))

# print("\nKept technology mix:")
# print(kept_rows['technology_type'].value_counts(normalize=True))

# PGE_intercon = kept_rows.copy()


In [16]:
# over_1mw = PGE_intercon[PGE_intercon['total_capacity_kw'] >= 1000]
# print(over_1mw)
# print(len(over_1mw))

# print(over_1mw["sector"].value_counts())

# PGE_intercon[PGE_intercon['total_capacity_kw'] <= 1000 | PGE_intercon['sector'] == "Residential"]

# Tracking the Sun - Solar PV Data

In [17]:
tracking_the_sun = pd.read_csv("../data/raw/solar/TTS_LBNL_public_file_29-Sep-2025_all.csv")
tracking_the_sun["zip_code"] = tracking_the_sun["zip_code"].astype(str).str.zfill(5).str.strip().str[:5]
tracking_the_sun = tracking_the_sun[tracking_the_sun['state'] == "CA"]
tracking_the_sun = tracking_the_sun[
    (tracking_the_sun['zip_code'] != "-0001")
    & (tracking_the_sun['zip_code'] != "-01.0")
    & (tracking_the_sun['zip_code'] != "000.0")
    & (tracking_the_sun['zip_code'] != "2399.")
    & (tracking_the_sun['zip_code'] != "831.0")]

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_94194/2920315716.py:1: DtypeWarning: Columns (1,2,3,11,15,16,18,28,29,31,32,34,35,38,39,40,53,54,56,57,59,60,74,75,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  tracking_the_sun = pd.read_csv("../data/raw/solar/TTS_LBNL_public_file_29-Sep-2025_all.csv")


# Combining the Datasets

In [18]:
# List of dataframes and their names for prefixing
datasets = {
    "tracking_the_sun": tracking_the_sun,
    # "SDGE_intercon": SDGE_intercon,
    # "SCE_intercon": SCE_intercon,
    # "PGE_intercon": PGE_intercon,
    "ev_chargers": ev_chargers,
    "power_plant_der": power_plant_der,
    "uswtdb_wind": uswtdb_wind,
    "storage_der": storage_der,
}

# Standardize and prefix columns (except for 'zip_code')
for name, df in datasets.items():
    df.to_csv(f'../data/processed/{name}.csv', index=True)

# Now `merged_df` contains the combined result

In [19]:
# EV Chargers
aggregated_ev_chargers = ev_chargers.groupby("zip_code")["total_chargers"].sum().reset_index()
aggregated_ev_chargers["level1_chargers"] = ev_chargers.groupby("zip_code")["level1_chargers"].sum().reset_index()['level1_chargers']
aggregated_ev_chargers["level2_chargers"] = ev_chargers.groupby("zip_code")["level2_chargers"].sum().reset_index()['level2_chargers']
aggregated_ev_chargers["dc_fast_chargers"] = ev_chargers.groupby("zip_code")["dc_fast_chargers"].sum().reset_index()['dc_fast_chargers']
aggregated_ev_chargers["zip_code"] = aggregated_ev_chargers["zip_code"].astype(str).str.zfill(5).str.strip()

inter = ev_cars.groupby("zip_code")
aggregated_ev_cars = inter.size().rename("zev_count").to_frame()
aggregated_ev_cars = aggregated_ev_cars.reset_index()
aggregated_ev_cars["zip_code"] = aggregated_ev_cars["zip_code"].astype(str).str.zfill(5).str.strip()


# TTS
tracking_the_sun = tracking_the_sun[tracking_the_sun["PV_system_size_DC"] >= 0]
tracking_the_sun["PV_system_size_DC"] = tracking_the_sun["PV_system_size_DC"] / 1000
aggregated_TTS = tracking_the_sun.groupby("zip_code")["PV_system_size_DC"].sum().reset_index()
aggregated_TTS["zip_code"] = aggregated_TTS["zip_code"].astype(str).str.zfill(5).str.strip()


# Power Plant
aggregated_power_plant = power_plant_der.groupby("zip_code")["plant_capacity_mw"].sum().reset_index()
aggregated_power_plant["zip_code"] = aggregated_power_plant["zip_code"].astype(str).str.zfill(5).str.strip()


# Storage
aggregated_storage = storage_der.groupby("zip_code")["storage_capacity_mw"].sum().reset_index()
aggregated_storage["zip_code"] = aggregated_storage["zip_code"].astype(str).str.zfill(5).str.strip()


# Wind
aggregated_wind = wind_zip.copy()
aggregated_wind["zip_code"] = aggregated_wind["zip_code"].astype(str).str.zfill(5).str.strip()


aggregated_datasets = [aggregated_TTS,
                       aggregated_ev_chargers,
                       aggregated_ev_cars,
                       aggregated_power_plant,
                       aggregated_storage,
                       aggregated_wind]
aggregated_names = ["aggregated_TTS",
                       "aggregated_ev_chargers",
                       "aggregated_ev_cars",
                       "aggregated_power_plant",
                       "aggregated_storage",
                       "aggregated_wind"]
for df, name in zip(aggregated_datasets, aggregated_names):
    df.to_csv(f'../data/processed/{name}.csv', index=True)

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_94194/834757802.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tracking_the_sun["PV_system_size_DC"] = tracking_the_sun["PV_system_size_DC"] / 1000


In [20]:
print(aggregated_ev_chargers.describe())
print(aggregated_ev_cars.describe())
print(aggregated_TTS.describe())
print(aggregated_power_plant.describe())
print(aggregated_wind.describe())
print(aggregated_storage.describe())

       total_chargers  level1_chargers  level2_chargers  dc_fast_chargers
count     1258.000000      1258.000000      1258.000000       1258.000000
mean       141.930843         0.550079       128.917329         13.013514
std       3386.105188        12.034931      3323.864667         64.460016
min          1.000000         0.000000         0.000000          0.000000
25%          8.000000         0.000000         4.000000          0.000000
50%         22.000000         0.000000        13.000000          4.000000
75%         53.000000         0.000000        37.000000         16.000000
max     120097.000000       420.000000    117884.000000       2213.000000
         zev_count
count  2293.000000
mean    138.407763
std     223.916265
min       1.000000
25%       8.000000
50%      59.000000
75%     266.000000
max    8418.000000
       PV_system_size_DC
count        2133.000000
mean            9.152211
std            15.753105
min             0.000660
25%             0.056700
50%          

In [21]:
merged_df = aggregated_datasets[0]
for df in aggregated_datasets[1:]:
    merged_df = pd.merge(merged_df, df, on="zip_code", how='outer')

In [22]:
merged_df = merged_df[merged_df["zip_code"].notna()]
merged_df.describe()
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2602 entries, 0 to 2601
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   zip_code             2602 non-null   object 
 1   PV_system_size_DC    2145 non-null   float64
 2   total_chargers       1258 non-null   float64
 3   level1_chargers      1258 non-null   float64
 4   level2_chargers      1258 non-null   float64
 5   dc_fast_chargers     1258 non-null   float64
 6   zev_count            2305 non-null   float64
 7   plant_capacity_mw    127 non-null    float64
 8   storage_capacity_mw  1597 non-null   float64
 9   wind_capacity_mw     42 non-null     float64
 10  wind_turbine_count   42 non-null     float64
dtypes: float64(10), object(1)
memory usage: 223.7+ KB


In [23]:
merged_df = merged_df.drop([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
merged_df = merged_df[(merged_df["zip_code"] != 'No in') & (merged_df["zip_code"] != '9227.')]
merged_df.to_csv(f'../data/processed/combined_der_dataset_full.csv', index=True)
merged_df.head(20)

,zip_code,PV_system_size_DC,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,zev_count,plant_capacity_mw,storage_capacity_mw,wind_capacity_mw,wind_turbine_count
14,90001,3.954350,1.0,0.0,1.0,0.0,166.0,NaN,0.28608,NaN,NaN
15,90002,2.109484,11.0,0.0,9.0,2.0,133.0,NaN,0.09880,NaN,NaN
16,90003,1.013024,16.0,0.0,16.0,0.0,178.0,NaN,0.11276,NaN,NaN
17,90004,1.616081,12.0,0.0,10.0,2.0,371.0,NaN,0.72647,NaN,NaN
18,90005,0.904518,28.0,0.0,28.0,0.0,260.0,NaN,0.12880,NaN,NaN
19,90006,0.586425,22.0,0.0,22.0,0.0,243.0,NaN,0.02200,NaN,NaN
20,90007,1.754401,211.0,0.0,211.0,0.0,174.0,NaN,0.01884,NaN,NaN
21,90008,1.612915,10.0,0.0,10.0,0.0,308.0,NaN,0.47226,NaN,NaN
22,90009,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN
23,90010,1.604140,13.0,0.0,7.0,6.0,161.0,NaN,NaN,NaN,NaN


# Combining with Predictors and Controls

In [6]:
df = pd.read_csv('../data/processed/combined_der_dataset_full.csv')
df["zip_code"] = df["zip_code"].astype(int)
acs = pd.read_csv('../data/processed/acs_predictors_ca_zip.csv')
ghi = pd.read_csv('../data/processed/ca_zip_ghi_mean_2023.csv')
temp = pd.read_csv('../data/processed/ca_zip_temperature_controls_2023.csv')
wind = pd.read_csv('../data/processed/ca_zip_wind_means_2023.csv')
zip2utility = pd.read_csv('../data/processed/zip_to_utility.csv')
demand = pd.read_csv('../data/processed/demand.csv')
energy_burden = pd.read_csv('../data/processed/energy_burden.csv')

for d in [acs, ghi, temp, wind, zip2utility, demand, energy_burden]:
    d.drop(columns=['Unnamed: 0'], inplace=True, errors="ignore")
    df = pd.merge(df, d, on="zip_code", how='left')

der_zero_cols = [
    "PV_system_size_DC",
    "total_chargers",
    "level1_chargers",
    "level2_chargers",
    "dc_fast_chargers",
    "zev_count",
    "plant_capacity_mw",
    "storage_capacity_mw",
    "wind_capacity_mw",
    "wind_turbine_count",
]
for col in der_zero_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
df.drop(columns = ["Unnamed: 0_x", "Unnamed: 0_y",
                   "lat_x", "lon_x", "lat_y", "lon_y"],
                   inplace = True,
                   errors="ignore")

ZCTA_SHP = "../data/raw/boundaries/tl_2023_us_zcta520/tl_2023_us_zcta520.shp"
COUNTY_SHP = "../data/raw/boundaries/tl_2023_us_county/tl_2023_us_county.shp"
zcta = gpd.read_file(ZCTA_SHP)
county = gpd.read_file(COUNTY_SHP)
zcta_id = "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else ("GEOID20" if "GEOID20" in zcta.columns else "GEOID")
county_id = "GEOID" if "GEOID" in county.columns else ("GEOID20" if "GEOID20" in county.columns else "GEOID10")
county_name_col = "NAME" if "NAME" in county.columns else None
zcta = zcta[[zcta_id, "geometry"]].rename(columns={zcta_id: "zip_code"})
keep = [county_id, "geometry"] + ([county_name_col] if county_name_col else [])
county = county[keep].rename(columns={county_id: "county_geoid"})
if county_name_col:
    county = county.rename(columns={county_name_col: "county_name"})
zcta["zip_code"] = zcta["zip_code"].astype(str).str.zfill(5)
zcta = zcta.to_crs("EPSG:5070")
county = county.to_crs("EPSG:5070")
pairs = gpd.sjoin(zcta, county, how="inner", predicate="intersects").drop(columns=["index_right"])
county_geom = county.set_index("county_geoid").geometry
other = gpd.GeoSeries(pairs["county_geoid"].map(county_geom), index=pairs.index, crs=pairs.crs)
pairs["overlap_area"] = pairs.geometry.intersection(other).area
pairs = pairs.sort_values(["zip_code", "overlap_area"], ascending=[True, False])
zip_to_county = pairs.drop_duplicates("zip_code")[["zip_code", "county_geoid"] + (["county_name"] if "county_name" in pairs.columns else [])]
df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
df = df.merge(zip_to_county, on="zip_code", how="left")
print(df[["zip_code", "county_geoid"] + (["county_name"] if "county_name" in df.columns else [])].head())

zcta = gpd.read_file(ZCTA_SHP)
zcta_id = "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else "GEOID20"
zcta = zcta[[zcta_id, "geometry"]].rename(columns={zcta_id: "zip_code"})
zcta["zip_code"] = zcta["zip_code"].astype(str).str.zfill(5)
zcta = zcta.to_crs("EPSG:5070")
zcta["area_km2"] = zcta.geometry.area / 1e6
df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
df = df.merge(zcta[["zip_code", "area_km2"]], on="zip_code", how="left")
df["pop_density_km2"] = df["total_population"] / df["area_km2"].replace(0, np.nan)
df["log_pop_density"] = np.log1p(df["pop_density_km2"])
min_n = 5
df["county_geoid"] = df["county_geoid"].astype(str).str.strip()
counts = df["county_geoid"].value_counts()
keep = counts[counts >= min_n].index
df = df[df["county_geoid"].isin(keep)].copy()
df.to_csv(f'../data/processed/combined_der_dataset_w_controls_predictors.csv', index=True)
print(df.info())

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_4050/3772371861.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["wind_capacity_mw"].fillna(0, inplace = True)
/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_4050/3772371861.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

  zip_code county_geoid  county_name
0    90001        06037  Los Angeles
1    90002        06037  Los Angeles
2    90003        06037  Los Angeles
3    90004        06037  Los Angeles
4    90005        06037  Los Angeles
<class 'pandas.core.frame.DataFrame'>
Index: 2545 entries, 0 to 2557
Data columns (total 46 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  2545 non-null   int64  
 1   zip_code                    2545 non-null   object 
 2   PV_system_size_DC           2106 non-null   float64
 3   total_chargers              1236 non-null   float64
 4   level1_chargers             1236 non-null   float64
 5   level2_chargers             1236 non-null   float64
 6   dc_fast_chargers            1236 non-null   float64
 7   zev_count                   2285 non-null   float64
 8   plant_capacity_mw           2545 non-null   float64
 9   storage_capacity_mw         1582 non-null   